# Liver Segmentation & Lesion Analysis Pipeline — Demo

This notebook is a thin demonstration/research layer over the reusable
implementation in `app/`. It does **not** duplicate the segmentation or
analysis logic — it imports and calls the same functions used by
`app/main.py`.

The dataset layout expected below matches the original research dataset:

```
CT_scans/
  <optional_category_folder>/
    case1.nii.gz
    case2.nii.gz
  ...
```

`<category_folder>` (e.g. `liver_lesion`, `normal`, `abnormal_no_liver`) is
carried through purely as optional metadata for dataset organization and
evaluation. It has no effect on what is processed or reported — every case
is analyzed the same way, and lesions are reported wherever the
segmentation actually finds them.

In [ ]:
from pathlib import Path

from app.pipeline.case_processor import discover_cases, process_directory
from app.reporting.summary_export import build_summary_dataframe, export_summary, split_by_zero_volume
from app.segmentation.config import SegmentationConfig
from app.utils.logging_config import configure_logging

configure_logging()

INPUT_DIR = Path("CT_scans")
OUTPUT_DIR = Path("liver_seg_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 1. Discover cases

Category (if present as a subfolder name) is attached as metadata only.

In [ ]:
cases = discover_cases(INPUT_DIR)
print(f"Found {len(cases)} case(s).")
cases[0] if cases else None

## 2. Run the pipeline

Each case is segmented (TotalSegmentator `liver_segments` + `liver_lesions`) and analyzed independently. This can take a while on first run per case; cached `statistics.json` files are reused on subsequent runs.

In [ ]:
results = process_directory(INPUT_DIR, OUTPUT_DIR, SegmentationConfig())
for r in results[:5]:
    print(r.ct_scan.study_id, "->", r.status.value, "-", (r.analysis.summary if r.analysis else r.error_message))

## 3. Build & export the summary table

In [ ]:
summary_df = build_summary_dataframe(results)
summary_df

In [ ]:
summary_csv = export_summary(results, OUTPUT_DIR)
print(f"Saved summary to: {summary_csv.resolve()}")

## 4. Zero-volume triage

Splits cases into those with any zero-volume measurement (worth reviewing) vs. clean cases.

In [ ]:
zero_csv, non_zero_csv = split_by_zero_volume(summary_df, OUTPUT_DIR)
print(f"Cases with zero volume:    {zero_csv}")
print(f"Cases without zero volume: {non_zero_csv}")

## Notes

- This notebook is a demo/experimentation layer. All segmentation and
  analysis logic lives in `app/` and is covered by tests in `tests/`.
- Case status (`VALID_NO_LESION`, `VALID_WITH_LESION`, `SEGMENTATION_FAILED`,
  `ANALYSIS_FAILED`) is derived entirely from segmentation output, never
  from the dataset folder name.
- Findings here are research-prototype output for review by a radiologist,
  not a clinical diagnosis.